In [1]:
# %% [1] 피처 계산 (한 번만 실행, 이후 재사용)
import pandas as pd
import numpy as np
import joblib
import time
from rdkit import Chem
from rdkit.Chem import Descriptors, rdFingerprintGenerator
from sklearn.model_selection import RandomizedSearchCV, cross_val_score
from lightgbm import LGBMClassifier

morgan_gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
hazard_top40 = joblib.load("./models/hazard_top40_descriptors.joblib")

def get_features(smiles, descriptor_names=hazard_top40):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    fp = np.array(morgan_gen.GetFingerprint(mol))
    all_d = Descriptors.CalcMolDescriptors(mol)
    d = np.array([all_d.get(name, 0.0) for name in descriptor_names], dtype=float)
    d = np.nan_to_num(d, nan=0.0, posinf=0.0, neginf=0.0)
    d = np.clip(d, -1e6, 1e6)
    return np.concatenate([fp, d])

tox21 = pd.read_csv("./data/hazard/moleculenet_tox21.csv")
print("피처 계산 중...")
t0 = time.time()
features = np.array([get_features(s) for s in tox21["SMILES"]])
print(f"완료 ({time.time()-t0:.0f}초)")

피처 계산 중...


완료 (35초)


In [2]:
# %% [2] 촘촘한 그리드로 튜닝 (n_iter 20 -> 50, 파라미터 종류 추가)
lgbm_param_dist_dense = {
    "n_estimators": [100, 200, 300, 500, 800],
    "num_leaves": [15, 31, 47, 63, 95],
    "learning_rate": [0.005, 0.01, 0.03, 0.05, 0.1],
    "min_child_samples": [5, 10, 20, 30, 50],
    "subsample": [0.6, 0.8, 1.0],
    "colsample_bytree": [0.6, 0.8, 1.0],
    "reg_alpha": [0.0, 0.1, 1.0],
    "reg_lambda": [0.0, 0.1, 1.0],
}

TARGET_TASKS = ["NR-ER-LBD", "SR-MMP"]
final_models = {}

for task in TARGET_TASKS:
    t0 = time.time()
    mask = tox21[task].notna()
    X = features[mask.values]
    y = tox21.loc[mask, task].values

    default_model = LGBMClassifier(class_weight="balanced", random_state=42, verbose=-1)
    default_score = cross_val_score(default_model, X, y, cv=5, scoring="roc_auc", n_jobs=-1).mean()

    search = RandomizedSearchCV(
        LGBMClassifier(class_weight="balanced", random_state=42, verbose=-1),
        param_distributions=lgbm_param_dist_dense,
        n_iter=50, cv=5, scoring="roc_auc", random_state=42, n_jobs=-1, verbose=1,
    )
    search.fit(X, y)

    if search.best_score_ > default_score:
        final_models[task] = search.best_estimator_
        winner = f"튜닝 채택 (AUC={search.best_score_:.4f} > 기본값 {default_score:.4f})"
    else:
        final_models[task] = default_model.fit(X, y)
        winner = f"기본값 유지 (기본값={default_score:.4f} >= 튜닝 {search.best_score_:.4f})"

    print(f"{task}: {winner}, {time.time()-t0:.0f}초, best_params={search.best_params_}")

joblib.dump(final_models, "./models/hazard_lgbm_tuned.joblib")
print("\n저장 완료: ./models/hazard_lgbm_tuned.joblib")

Fitting 5 folds for each of 50 candidates, totalling 250 fits
NR-ER-LBD: 튜닝 채택 (AUC=0.8457 > 기본값 0.8357), 168초, best_params={'subsample': 1.0, 'reg_lambda': 0.1, 'reg_alpha': 0.1, 'num_leaves': 63, 'n_estimators': 300, 'min_child_samples': 5, 'learning_rate': 0.1, 'colsample_bytree': 0.8}
Fitting 5 folds for each of 50 candidates, totalling 250 fits
SR-MMP: 튜닝 채택 (AUC=0.9280 > 기본값 0.9211), 166초, best_params={'subsample': 0.8, 'reg_lambda': 1.0, 'reg_alpha': 0.1, 'num_leaves': 63, 'n_estimators': 500, 'min_child_samples': 5, 'learning_rate': 0.05, 'colsample_bytree': 0.6}

저장 완료: ./models/hazard_lgbm_tuned.joblib


In [1]:
import pandas as pd
import numpy as np
import joblib
import time
from rdkit import Chem
from rdkit.Chem import Descriptors, rdFingerprintGenerator
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestClassifier

morgan_gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)

def get_features(smiles, descriptor_names):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    fp = np.array(morgan_gen.GetFingerprint(mol))
    all_d = Descriptors.CalcMolDescriptors(mol)
    d = np.array([all_d.get(name, 0.0) for name in descriptor_names], dtype=float)
    d = np.nan_to_num(d, nan=0.0, posinf=0.0, neginf=0.0)
    d = np.clip(d, -1e6, 1e6)
    return np.concatenate([fp, d])

# ---------- 1. bp 모델 (기본값 HistGB, bp 전용 top40) ----------
print("=== 1. bp 모델 ===")
t0 = time.time()
bp_top40 = joblib.load("./models/bp_top40_descriptors.joblib")
bp = pd.read_csv("./data/processed/crc_bp_c.csv")
X_bp = np.array([get_features(s, bp_top40) for s in bp["SMILES"]])
y_bp = bp["Processed tbp/ºC"].values

cv_r2 = cross_val_score(HistGradientBoostingRegressor(random_state=42), X_bp, y_bp, cv=5, scoring="r2", n_jobs=-1)
print(f"bp 기본값 5-fold CV R2 = {cv_r2.mean():.4f} (+-{cv_r2.std():.4f})")

bp_model = HistGradientBoostingRegressor(random_state=42).fit(X_bp, y_bp)
joblib.dump(bp_model, "./models/bp_model.joblib")
print(f"bp_model.joblib 저장 완료 ({time.time()-t0:.0f}초)\n")

# ---------- 2. 유해성 10개 태스크 (기본값 RandomForest, 유해성 전용 top40) ----------
print("=== 2. 유해성 10개 태스크 (RandomForest 기본값) ===")
t0 = time.time()
hazard_top40 = joblib.load("./models/hazard_top40_descriptors.joblib")
tox21 = pd.read_csv("./data/hazard/moleculenet_tox21.csv")
X_tox21 = np.array([get_features(s, hazard_top40) for s in tox21["SMILES"]])

all_tasks = [c for c in tox21.columns if c not in ("SMILES", "InChIKey")]
LGBM_TASKS = ["NR-ER-LBD", "SR-MMP"]  # 이미 튜닝 완료된 애들 제외
RF_TASKS = [t for t in all_tasks if t not in LGBM_TASKS]

rf_models = {}
for task in RF_TASKS:
    tt = time.time()
    mask = tox21[task].notna()
    X = X_tox21[mask.values]
    y = tox21.loc[mask, task].values

    cv_auc = cross_val_score(
        RandomForestClassifier(class_weight="balanced", random_state=42),
        X, y, cv=5, scoring="roc_auc", n_jobs=-1,
    )
    model = RandomForestClassifier(class_weight="balanced", random_state=42).fit(X, y)
    rf_models[task] = model
    print(f"{task}: AUC={cv_auc.mean():.4f} (+-{cv_auc.std():.4f}), {time.time()-tt:.0f}초")

print(f"10개 완료 ({time.time()-t0:.0f}초)\n")

# ---------- 3. 최종 tox21_models.joblib 조립 (RF 10개 + 튜닝된 LightGBM 2개) ----------
print("=== 3. 최종 모델 조립 ===")
lgbm_tuned = joblib.load("./models/hazard_lgbm_tuned.joblib")  # {"NR-ER-LBD":..., "SR-MMP":...}
tox21_models = {**rf_models, **lgbm_tuned}
joblib.dump(tox21_models, "./models/tox21_models.joblib")
print(f"tox21_models.joblib 저장 완료 (총 {len(tox21_models)}개 태스크)")

=== 1. bp 모델 ===
bp 기본값 5-fold CV R2 = 0.9194 (+-0.0198)
bp_model.joblib 저장 완료 (43초)

=== 2. 유해성 10개 태스크 (RandomForest 기본값) ===


NR-AR: AUC=0.8109 (+-0.0078), 4초
NR-AR-LBD: AUC=0.8796 (+-0.0317), 4초
NR-AhR: AUC=0.9069 (+-0.0093), 4초
NR-Aromatase: AUC=0.8609 (+-0.0190), 2초
NR-ER: AUC=0.7313 (+-0.0096), 3초
NR-PPAR-gamma: AUC=0.8439 (+-0.0338), 2초
SR-ARE: AUC=0.8326 (+-0.0076), 3초
SR-ATAD5: AUC=0.8544 (+-0.0126), 2초
SR-HSE: AUC=0.8099 (+-0.0270), 3초
SR-p53: AUC=0.8656 (+-0.0092), 3초
10개 완료 (65초)

=== 3. 최종 모델 조립 ===
tox21_models.joblib 저장 완료 (총 12개 태스크)
